### Initialisation

In [1]:
import pandas as pd
import duckdb
from pathlib import Path

### Helper Functions

In [2]:
# Helpers for safely quoting column names in SQL
def quote_ident(name):
    return '"' + name.replace('"', '""') + '"'

### Exploratory Data Analysis

In [3]:
CSV_FILE = Path.cwd() / "usdot_insurance_events_2026-10-12.csv"

if not CSV_FILE.exists():
    raise FileNotFoundError(f"CSV not found: {CSV_FILE}")

print(f"Dataset: {CSV_FILE}")
print(f"File size: {CSV_FILE.stat().st_size / 1024**3:.2f} GB")

# Use an in-memory DuckDB database. The source CSV stays untouched.
con = duckdb.connect(":memory:")
con.execute("SET memory_limit = '6GB'")
con.execute("SET threads = 4")

# Use a temporary directory only if DuckDB needs to spill data.
# This does not create a copy of the source CSV.
temp_dir = Path.cwd() / "duckdb_temp"
temp_dir.mkdir(exist_ok=True)
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")

csv_path = CSV_FILE.resolve().as_posix().replace("'", "''")

con.execute(f"""
    CREATE VIEW raw AS
    SELECT *
    FROM read_csv_auto(
        '{csv_path}',
        sample_size = 100000
    )
""")

Dataset: c:\Users\hi\Documents\ROSELLA ASSIGNMENT\usdot_insurance_events_2026-10-12.csv
File size: 43.87 GB


In [4]:
# --------------------------------------------------
# 1. Schema: column names and inferred data types
# --------------------------------------------------

print("\n" + "=" * 80)
print("1. SCHEMA")
print("=" * 80)

schema = con.execute("DESCRIBE raw").df()
print(schema.to_string(index=False))

columns = schema["column_name"].tolist()
numeric_types = {
    "TINYINT", "SMALLINT", "INTEGER", "BIGINT", "HUGEINT",
    "UTINYINT", "USMALLINT", "UINTEGER", "UBIGINT",
    "FLOAT", "REAL", "DOUBLE",
    "DECIMAL", "NUMERIC"
}

numeric_columns = [
    row["column_name"]
    for _, row in schema.iterrows()
    if any(
        str(row["column_type"]).upper().startswith(t)
        for t in numeric_types
    )
]

print(f"\nTotal columns: {len(columns)}")
print(f"Numeric columns: {len(numeric_columns)}")


1. SCHEMA
                                     column_name column_type null  key default extra
                                           usdot      BIGINT  YES None    None  None
                                    company_name     VARCHAR  YES None    None  None
                                        dba_name     VARCHAR  YES None    None  None
                                           state     VARCHAR  YES None    None  None
                             has_insurance_event     BOOLEAN  YES None    None  None
                                    event_source     VARCHAR  YES None    None  None
                          event_usdot_matched_by     VARCHAR  YES None    None  None
                                    event_status     VARCHAR  YES None    None  None
                                     policy_line     VARCHAR  YES None    None  None
                                         insurer     VARCHAR  YES None    None  None
                                   insurer_group     V

In [5]:
# --------------------------------------------------
# 2. Sample records
# --------------------------------------------------

print("\n" + "=" * 80)
print("2. FIRST 10 ROWS")
print("=" * 80)

print(con.execute("SELECT * FROM raw LIMIT 10").df().to_string(index=False))


2. FIRST 10 ROWS
 usdot                                company_name                  dba_name state  has_insurance_event event_source event_usdot_matched_by event_status policy_line insurer insurer_group coverage_class form_code class_code policy_number docket_or_certificate_number effective_date expiration_date cancellation_or_end_date end_reason cancellation_form_code filed_or_received_date rejection_reason  event_year years_covered all_insurers_this_usdot_in_event_year  insurer_count_this_usdot_in_event_year  coverage_amount_usd  underlying_limit_usd  min_required_coverage_usd  premium_usd rating_state  vehicle_count insurer_branch agent_name  is_peo  residual_market event_details_json event_company_signature  insurance_event_count  insurance_distinct_insurers insurance_first_event_date insurance_latest_event_date insurance_all_insurers insurance_active_bipd_insurers_as_of_2026_05_26 insurance_active_cargo_insurers_as_of_2026_05_26 insurance_wc_insurers  insurance_cancelled_count  

In [6]:
# --------------------------------------------------
# 3. Total row count
# --------------------------------------------------

print("\n" + "=" * 80)
print("3. TOTAL ROWS")
print("=" * 80)

row_count = con.execute(
    "SELECT COUNT(*) FROM raw"
).fetchone()[0]

print(f"Rows: {row_count:,}")


3. TOTAL ROWS
Rows: 12,243,741


In [7]:
expressions = []

for i, col in enumerate(columns):
    c = quote_ident(col)
    expressions.append(f"COUNT({c}) AS nonnull_{i}")

query = "SELECT " + ", ".join(expressions) + " FROM raw"
result = con.execute(query).fetchone()

profile = []

for i, col in enumerate(columns):
    nonnull = result[i]
    profile.append({
        "column": col,
        "null_count": row_count - nonnull,
        "null_pct": (
            (row_count - nonnull) / row_count * 100
            if row_count else 0
        )
    })

profile_df = pd.DataFrame(profile).sort_values(
    "null_pct", ascending=False
)

print(profile_df.to_string(index=False))

                                          column  null_count   null_pct
                                    co_sic_codes    12243741 100.000000
                                       co_county    12243741 100.000000
                               census_trplimo_16    12243641  99.999183
                              census_mail_barrio    12243601  99.998857
                           census_trpschool_9_15    12243594  99.998799
                            census_trpschool_1_8    12243568  99.998587
                               census_total_cars    12243558  99.998505
                             census_trplimo_9_15    12243482  99.997885
                             census_trpschool_16    12243400  99.997215
                              census_trplimo_1_8    12243384  99.997084
                               census_phy_barrio    12243310  99.996480
                            census_trmschool_1_8    12243202  99.995598
                  census_prior_revoke_dot_number    12243157  99

In [8]:
key_columns = [
    # Company identity
    "usdot",
    "company_name",
    "state",

    # Insurance event details
    "has_insurance_event",
    "event_source",
    "event_usdot_matched_by",
    "event_status",
    "event_year",
    "policy_line",
    "insurer",
    "insurer_group",
    "coverage_class",
    "policy_number",

    # Coverage and policy lifecycle
    "effective_date",
    "expiration_date",
    "cancellation_or_end_date",
    "end_reason",
    "filed_or_received_date",
    "rejection_reason",
    "coverage_amount_usd",
    "min_required_coverage_usd",
    "premium_usd",

    # Insurance history summaries
    "insurance_event_count",
    "insurance_distinct_insurers",
    "insurance_cancelled_count",
    "insurance_replaced_count",
    "insurance_rejected_count",

    # Fleet and safety profile
    "census_power_units",
    "census_total_drivers",
    "crash_count",
    "inspection_count",
    "revocation_count",
    "oos_order_count",

    # Business enrichment
    "co_primary_naics",
    "co_years_in_business",
    "co_employee_band",
    "co_match_confidence",
]

expressions = [
    f"APPROX_COUNT_DISTINCT({quote_ident(col)}) AS {quote_ident(col)}"
    for col in key_columns
]

query = "SELECT " + ", ".join(expressions) + " FROM raw"
print(con.execute(query).fetchdf())

     usdot  company_name  state  has_insurance_event  event_source  \
0  4205801       4222370    117                    2            13   

   event_usdot_matched_by  event_status  event_year  policy_line  insurer  \
0                       3            12          85            9     6453   

   ...  census_power_units  census_total_drivers  crash_count  \
0  ...                2577                  1722          266   

   inspection_count  revocation_count  oos_order_count  co_primary_naics  \
0               932                27                6              1420   

   co_years_in_business  co_employee_band  co_match_confidence  
0                   112                10                    2  

[1 rows x 37 columns]


In [9]:
# --------------------------------------------------
# 5. DuckDB's built-in descriptive summary
# Includes statistics appropriate to each column type
# --------------------------------------------------
print("\n" + "=" * 80)
print("5. DESCRIPTIVE SUMMARY")
print("=" * 80)

summary = con.execute("SUMMARIZE raw").df()
print(summary.to_string(index=False))


5. DESCRIPTIVE SUMMARY
                                     column_name column_type                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                        min                         

In [10]:
# --------------------------------------------------
# 6. Numeric statistics
# --------------------------------------------------

if numeric_columns:
    print("\n" + "=" * 80)
    print("6. NUMERIC STATISTICS")
    print("=" * 80)

    expressions = []

    for i, col in enumerate(numeric_columns):
        c = quote_ident(col)

        expressions.extend([
            f"MIN({c}) AS min_{i}",
            f"MAX({c}) AS max_{i}",
            f"AVG(TRY_CAST({c} AS DOUBLE)) AS mean_{i}",
            f"STDDEV_SAMP(TRY_CAST({c} AS DOUBLE)) AS std_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.25) AS p25_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.50) AS median_{i}",
            f"APPROX_QUANTILE(TRY_CAST({c} AS DOUBLE), 0.75) AS p75_{i}"
        ])

    values = con.execute(
        "SELECT " + ", ".join(expressions) + " FROM raw"
    ).fetchone()

    numeric_profile = []

    for i, col in enumerate(numeric_columns):
        numeric_profile.append({
            "column": col,
            "min": values[i * 7],
            "max": values[i * 7 + 1],
            "mean": values[i * 7 + 2],
            "std": values[i * 7 + 3],
            "p25": values[i * 7 + 4],
            "median": values[i * 7 + 5],
            "p75": values[i * 7 + 6]
        })

    print(pd.DataFrame(numeric_profile).to_string(index=False))


6. NUMERIC STATISTICS
                                column           min          max          mean          std           p25        median           p75
                                 usdot  1.000000e+00 9.002022e+09  2.141387e+06 4.661490e+06  1.005510e+06  2.025180e+06  3.164565e+06
                            event_year  1.936000e+03 2.027000e+03  2.013818e+03 9.028065e+00  2.007000e+03  2.015421e+03  2.022000e+03
insurer_count_this_usdot_in_event_year  1.000000e+00 4.200000e+01  1.901650e+00 1.569668e+00  1.000000e+00  2.000000e+00  2.000000e+00
                   coverage_amount_usd  0.000000e+00 2.000000e+09  7.593126e+05 1.124660e+06  5.023452e+05  7.500000e+05  1.000000e+06
                  underlying_limit_usd  0.000000e+00 9.285800e+07  1.022706e+04 1.412732e+05  0.000000e+00  0.000000e+00  0.000000e+00
             min_required_coverage_usd  0.000000e+00 1.000000e+07  8.745397e+05 7.346829e+05  7.500000e+05  7.500000e+05  7.500000e+05
                           premi

In [12]:
query = """
WITH values_long AS (
    SELECT 'insurer' AS column_name,
           COALESCE(NULLIF(TRIM(CAST(insurer AS VARCHAR)), ''), '<NULL/BLANK>') AS value
    FROM raw

    UNION ALL
    SELECT 'insurer_group',
           COALESCE(NULLIF(TRIM(CAST(insurer_group AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'event_source',
           COALESCE(NULLIF(TRIM(CAST(event_source AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'event_status',
           COALESCE(NULLIF(TRIM(CAST(event_status AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'policy_line',
           COALESCE(NULLIF(TRIM(CAST(policy_line AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'coverage_class',
           COALESCE(NULLIF(TRIM(CAST(coverage_class AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'form_code',
           COALESCE(NULLIF(TRIM(CAST(form_code AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'class_code',
           COALESCE(NULLIF(TRIM(CAST(class_code AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'rating_state',
           COALESCE(NULLIF(TRIM(CAST(rating_state AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'all_insurers_this_usdot_in_event_year',
           COALESCE(NULLIF(TRIM(CAST(all_insurers_this_usdot_in_event_year AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_all_insurers',
           COALESCE(NULLIF(TRIM(CAST(insurance_all_insurers AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_active_bipd_insurers_as_of_2026_05_26',
           COALESCE(NULLIF(TRIM(CAST(insurance_active_bipd_insurers_as_of_2026_05_26 AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_active_cargo_insurers_as_of_2026_05_26',
           COALESCE(NULLIF(TRIM(CAST(insurance_active_cargo_insurers_as_of_2026_05_26 AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_wc_insurers',
           COALESCE(NULLIF(TRIM(CAST(insurance_wc_insurers AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw

    UNION ALL
    SELECT 'insurance_carriers_by_year',
           COALESCE(NULLIF(TRIM(CAST(insurance_carriers_by_year AS VARCHAR)), ''), '<NULL/BLANK>')
    FROM raw
),
freq AS (
    SELECT
        column_name,
        value,
        COUNT(*) AS frequency
    FROM values_long
    GROUP BY column_name, value
)
SELECT
    column_name,
    value,
    frequency,
    ROUND(
        100.0 * frequency /
        SUM(frequency) OVER (PARTITION BY column_name),
        3
    ) AS pct_of_rows
FROM freq
ORDER BY column_name, frequency DESC
"""

# Execute using your existing DuckDB connection
result = con.execute(query).fetchdf()

# Display the full distribution
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 150)

display(result)

RuntimeError: Query interrupted

In [13]:
categorical_columns = [
    "insurer",
    "insurer_group",
    "event_source",
    "event_status",
    "policy_line",
    "coverage_class",
    "form_code",
    "class_code",
    "rating_state",
]

values_clause = ",\n".join(
    f"('{col}', CAST({quote_ident(col)} AS VARCHAR))"
    for col in categorical_columns
)

query = f"""
WITH values_long AS (
    SELECT
        v.column_name,
        COALESCE(
            NULLIF(TRIM(v.value), ''),
            '<NULL/BLANK>'
        ) AS value
    FROM raw
    CROSS JOIN LATERAL (
        VALUES
            {values_clause}
    ) AS v(column_name, value)
),
freq AS (
    SELECT
        column_name,
        value,
        COUNT(*) AS frequency
    FROM values_long
    GROUP BY column_name, value
),
ranked AS (
    SELECT
        column_name,
        value,
        frequency,
        ROUND(
            100.0 * frequency /
            SUM(frequency) OVER (
                PARTITION BY column_name
            ),
            3
        ) AS pct_of_rows,
        ROW_NUMBER() OVER (
            PARTITION BY column_name
            ORDER BY frequency DESC
        ) AS rn
    FROM freq
)
SELECT
    column_name,
    value,
    frequency,
    pct_of_rows
FROM ranked
WHERE rn <= 50
ORDER BY column_name, frequency DESC
"""

result = con.execute(query).fetchdf()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 150)

display(result)

,column_name,value,frequency,pct_of_rows
0,class_code,<NULL/BLANK>,5953477,48.625
1,class_code,P,5905367,48.232
2,class_code,E,60480,0.494
3,class_code,5190,19960,0.163
4,class_code,3724,17804,0.145
5,class_code,5606,13566,0.111
6,class_code,7219,12226,0.100
7,class_code,5183,12133,0.099
8,class_code,8810,9713,0.079
9,class_code,5213,9686,0.079


In [1]:
import pandas 
import scikit-learn

SyntaxError: invalid syntax (2090533257.py, line 2)

In [ ]:
import duckdb
import pandas as pd
import numpy as np

from datetime import date
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

# ============================================================
# CONFIGURATION
# ============================================================

# Assumes your existing DuckDB connection is named `con`
# and your existing CSV view is named `raw`.

OUTPUT_FILE = "rosella_ranked_companies.csv"
TOP_N = 10000
MIN_CARRIER_COMPANIES = 30
MAX_TRAINING_ROWS_PER_YEAR = 50000

# These are candidate company-risk features.
# The code will use only columns that actually exist in `raw`.
CANDIDATE_FEATURES = [
    "vehicle_count",
    "power_units",
    "total_power_units",
    "driver_count",
    "total_drivers",
    "total_miles",
    "mileage",
    "fleet_size",
    "crash_total",
    "crash_count",
    "fatal_crash_count",
    "injury_crash_count",
    "tow_crash_count",
    "inspection_total",
    "inspection_count",
    "oos_order_count",
    "revocation_count",
    "safety_rating",
    "years_in_business",
    "coverage_amount_usd",
    "underlying_limit_usd",
]

# Fields that identify the event and target carrier.
REQUIRED_COLUMNS = [
    "usdot",
    "policy_line",
    "insurer",
    "effective_date",
    "filed_or_received_date",
    "event_status",
]

# ============================================================
# 0. CHECK THE DATASET SCHEMA
# ============================================================

schema = con.execute("DESCRIBE raw").df()
available_columns = set(schema["column_name"])

missing = [c for c in REQUIRED_COLUMNS if c not in available_columns]

if missing:
    raise ValueError(
        f"Missing expected columns: {missing}. "
        "Inspect con.execute('DESCRIBE raw').df() and update the code."
    )

feature_columns = [
    c for c in CANDIDATE_FEATURES
    if c in available_columns
]

if not feature_columns:
    print("No candidate features matched.")
    print("Available columns:")
    print(schema["column_name"].tolist())
    raise ValueError(
        "Update CANDIDATE_FEATURES with the actual company-risk column names."
    )

print("Features selected:", feature_columns)
print("Building the insurance-event timeline...")

# Quote identifiers safely for SQL.
def qi(name):
    return '"' + name.replace('"', '""') + '"'

# ============================================================
# 1. BUILD A CLEANER AUTO-LIABILITY EVENT TABLE
# ============================================================

# Materialise only the relevant event fields, rather than the
# full 376-column CSV. This makes subsequent operations cheaper.

con.execute("DROP TABLE IF EXISTS auto_events")

con.execute("""
CREATE TABLE auto_events AS
SELECT DISTINCT
    TRIM(CAST(usdot AS VARCHAR)) AS usdot,
    UPPER(TRIM(CAST(insurer AS VARCHAR))) AS insurer,
    LOWER(TRIM(CAST(event_status AS VARCHAR))) AS event_status,

    COALESCE(
        TRY_CAST(effective_date AS DATE),
        TRY_CAST(filed_or_received_date AS DATE)
    ) AS event_date

FROM raw

WHERE LOWER(TRIM(CAST(policy_line AS VARCHAR))) = 'auto liability'
  AND usdot IS NOT NULL
  AND TRIM(CAST(usdot AS VARCHAR)) <> ''
  AND insurer IS NOT NULL
  AND TRIM(CAST(insurer AS VARCHAR)) <> ''
  AND COALESCE(
        TRY_CAST(effective_date AS DATE),
        TRY_CAST(filed_or_received_date AS DATE)
      ) IS NOT NULL
""")

# This is a preliminary exclusion rule, not a definitive
# interpretation of the source's status semantics.
con.execute("DROP TABLE IF EXISTS eligible_events")

con.execute("""
CREATE TABLE eligible_events AS
SELECT *
FROM auto_events
WHERE event_status NOT LIKE '%cancel%'
  AND event_status NOT LIKE '%replac%'
  AND event_status NOT LIKE '%inactive%'
  AND event_status NOT LIKE '%reject%'
""")

date_info = con.execute("""
SELECT
    MIN(event_date) AS min_event_date,
    MAX(event_date) AS max_event_date,
    COUNT(*) AS event_rows,
    COUNT(DISTINCT usdot) AS distinct_companies,
    COUNT(DISTINCT insurer) AS distinct_insurers
FROM auto_events
""").df()

print("\nEVENT DATA SUMMARY")
print(date_info.to_string(index=False))

max_event_date = pd.Timestamp(date_info.loc[0, "max_event_date"]).date()

# ============================================================
# 2. BUILD COMPANY-LEVEL RISK FEATURES
# ============================================================

# These fields are expected to be company-level characteristics
# repeated across event rows. MAX ignores nulls and is appropriate
# only where the value is consistent across the company's rows.
#
# We will check feature quality below.

con.execute("DROP TABLE IF EXISTS company_features")

feature_selects = [
    f"MAX(TRY_CAST({qi(c)} AS DOUBLE)) AS {qi(c)}"
    for c in feature_columns
]

con.execute(f"""
CREATE TABLE company_features AS
SELECT
    TRIM(CAST(usdot AS VARCHAR)) AS usdot,
    {", ".join(feature_selects)}
FROM raw
WHERE usdot IS NOT NULL
  AND TRIM(CAST(usdot AS VARCHAR)) <> ''
GROUP BY TRIM(CAST(usdot AS VARCHAR))
""")

# Inspect missingness before trusting the feature set.
missingness = con.execute(
    "SELECT COUNT(*) AS n FROM company_features"
).df().iloc[0]["n"]

print(f"\nCompany feature table: {missingness:,} companies")

missing_rows = []
for c in feature_columns:
    result = con.execute(f"""
        SELECT
            COUNT(*) AS n,
            COUNT({qi(c)}) AS non_null,
            COUNT(DISTINCT {qi(c)}) AS distinct_values
        FROM company_features
    """).fetchone()

    missing_rows.append({
        "feature": c,
        "non_null": result[1],
        "null_pct": round(100 * (1 - result[1] / max(result[0], 1)), 2),
        "distinct_values": result[2],
    })

feature_quality = pd.DataFrame(missing_rows).sort_values("null_pct")
print("\nFEATURE QUALITY")
print(feature_quality.to_string(index=False))

# Keep features with at least some useful coverage and variation.
usable_features = feature_quality.loc[
    (feature_quality["null_pct"] < 95)
    & (feature_quality["distinct_values"] > 1),
    "feature"
].tolist()

if not usable_features:
    raise ValueError(
        "No usable numeric features remain. Review FEATURE QUALITY "
        "and add appropriate column names to CANDIDATE_FEATURES."
    )

print("\nFeatures used for carrier-fit scoring:", usable_features)

# ============================================================
# 3. IDENTIFY THE LATEST UNAMBIGUOUS PLACEMENT PER COMPANY
# ============================================================

# We exclude statuses that look cancelled, replaced, inactive or
# rejected. Blank statuses remain eligible. If multiple insurers
# occur on the latest eligible event date, we exclude that company
# from this initial scoring run rather than arbitrarily choosing one.

con.execute("DROP TABLE IF EXISTS latest_dates")
con.execute("""
CREATE TABLE latest_dates AS
SELECT usdot, MAX(event_date) AS latest_date
FROM eligible_events
GROUP BY usdot
""")

con.execute("DROP TABLE IF EXISTS current_placements")
con.execute("""
CREATE TABLE current_placements AS
SELECT
    e.usdot,
    MIN(e.insurer) AS insurer,
    d.latest_date AS placement_date,
    COUNT(DISTINCT e.insurer) AS carriers_on_latest_date
FROM eligible_events e
JOIN latest_dates d
  ON e.usdot = d.usdot
 AND e.event_date = d.latest_date
GROUP BY e.usdot, d.latest_date
HAVING COUNT(DISTINCT e.insurer) = 1
""")

placement_counts = con.execute("""
SELECT
    COUNT(*) AS unambiguous_companies,
    COUNT(DISTINCT insurer) AS carriers
FROM current_placements
""").df()

print("\nLATEST PLACEMENTS")
print(placement_counts.to_string(index=False))

# Attach company features to each company's latest placement.
con.execute("DROP TABLE IF EXISTS placement_features")

con.execute(f"""
CREATE TABLE placement_features AS
SELECT
    p.usdot,
    p.insurer,
    p.placement_date,
    {", ".join("f." + qi(c) for c in usable_features)}
FROM current_placements p
JOIN company_features f USING (usdot)
""")

# ============================================================
# 4. MODEL 1: CARRIER-FIT ANOMALY SCORE
# ============================================================

# Build a portfolio profile for each carrier.
# For each numeric feature, compare the company with its carrier's
# average and standard deviation.
#
# Carriers with fewer than MIN_CARRIER_COMPANIES observations are
# not scored using this approach, because their profiles may be
# too unreliable.

con.execute("DROP TABLE IF EXISTS carrier_profiles")

profile_aggregates = [
    f"AVG({qi(c)}) AS {qi('mean_' + c)}, "
    f"STDDEV_SAMP({qi(c)}) AS {qi('std_' + c)}"
    for c in usable_features
]

con.execute(f"""
CREATE TABLE carrier_profiles AS
SELECT
    insurer,
    COUNT(DISTINCT usdot) AS carrier_company_count,
    {", ".join(profile_aggregates)}
FROM placement_features
GROUP BY insurer
""")

# Compute average capped absolute z-score for each company.
# Capping at 3 prevents one extreme feature from dominating.

score_terms = []

for c in usable_features:
    score_terms.append(f"""
        CASE
          WHEN cp.carrier_company_count >= {MIN_CARRIER_COMPANIES}
           AND pf.{qi(c)} IS NOT NULL
           AND cp.{qi('std_' + c)} > 0
          THEN LEAST(
              3.0,
              ABS(
                (pf.{qi(c)} - cp.{qi('mean_' + c)})
                / cp.{qi('std_' + c)}
              )
          )
        END
    """)

numerator = " + ".join(
    f"COALESCE(({term}), 0)" for term in score_terms
)
denominator = " + ".join(
    f"CASE WHEN ({term}) IS NOT NULL THEN 1 ELSE 0 END"
    for term in score_terms
)

con.execute("DROP TABLE IF EXISTS carrier_fit_scores")

con.execute(f"""
CREATE TABLE carrier_fit_scores AS
WITH scored AS (
    SELECT
        pf.usdot,
        pf.insurer,
        pf.placement_date,
        cp.carrier_company_count,
        CASE
          WHEN cp.carrier_company_count >= {MIN_CARRIER_COMPANIES}
          THEN ({numerator}) / NULLIF(({denominator}), 0)
          ELSE NULL
        END AS fit_anomaly_score
    FROM placement_features pf
    JOIN carrier_profiles cp USING (insurer)
)
SELECT
    *,
    PERCENT_RANK() OVER (
        ORDER BY fit_anomaly_score
    ) AS fit_anomaly_percentile
FROM scored
""")

fit_summary = con.execute("""
SELECT
    COUNT(*) AS companies,
    COUNT(fit_anomaly_score) AS companies_scored,
    AVG(fit_anomaly_score) AS mean_anomaly_score,
    MAX(fit_anomaly_score) AS max_anomaly_score
FROM carrier_fit_scores
""").df()

print("\nMODEL 1 SUMMARY")
print(fit_summary.to_string(index=False))

# ============================================================
# 5. MODEL 2: BUILD HISTORICAL SNAPSHOTS
# ============================================================

# At each year-end, use only events observed up to that date.
# The label is 1 if a different eligible insurer appears during
# the following 365 days.
#
# IMPORTANT: this is a proxy for a genuine insurance switch.
# It can be wrong where policies overlap or event filings are noisy.

def snapshot_query(cutoff, include_label=True, sample=False):
    cutoff_sql = f"DATE '{cutoff.isoformat()}'"

    label_sql = ""
    if include_label:
        label_sql = f"""
        , CASE WHEN EXISTS (
            SELECT 1
            FROM eligible_events future
            WHERE future.usdot = cc.usdot
              AND future.event_date > {cutoff_sql}
              AND future.event_date <= {cutoff_sql} + INTERVAL '365 days'
              AND future.insurer <> cc.insurer
        ) THEN 1 ELSE 0 END AS future_switch
        """

    sample_sql = ""
    if sample:
        sample_sql = (
            f" ORDER BY HASH(usdot) LIMIT {MAX_TRAINING_ROWS_PER_YEAR}"
        )

    return f"""
    WITH hist AS (
        SELECT *
        FROM auto_events
        WHERE event_date <= {cutoff_sql}
    ),
    eligible_hist AS (
        SELECT *
        FROM eligible_events
        WHERE event_date <= {cutoff_sql}
    ),
    latest AS (
        SELECT usdot, MAX(event_date) AS latest_date
        FROM eligible_hist
        GROUP BY usdot
    ),
    latest_carriers AS (
        SELECT
            e.usdot,
            l.latest_date,
            MIN(e.insurer) AS insurer,
            COUNT(DISTINCT e.insurer) AS carrier_count
        FROM eligible_hist e
        JOIN latest l
          ON e.usdot = l.usdot
         AND e.event_date = l.latest_date
        GROUP BY e.usdot, l.latest_date
        HAVING COUNT(DISTINCT e.insurer) = 1
    ),
    company_stats AS (
        SELECT
            usdot,
            COUNT(*) AS event_count,
            COUNT(DISTINCT insurer) AS distinct_insurers,
            MIN(event_date) AS first_event_date,
            MAX(event_date) AS last_event_date,
            SUM(CASE WHEN event_status LIKE '%cancel%' THEN 1 ELSE 0 END)
                AS cancellation_count,
            SUM(CASE WHEN event_status LIKE '%replac%' THEN 1 ELSE 0 END)
                AS replacement_count,
            SUM(CASE WHEN event_status LIKE '%reject%' THEN 1 ELSE 0 END)
                AS rejection_count
        FROM hist
        GROUP BY usdot
    ),
    joined AS (
        SELECT
            cc.usdot,
            cc.insurer,
            YEAR({cutoff_sql}) AS snapshot_year,
            cs.event_count,
            cs.distinct_insurers,
            DATE_DIFF('day', cc.latest_date, {cutoff_sql})
                AS days_since_last_placement,
            DATE_DIFF('day', cs.first_event_date, cs.last_event_date) / 365.25
                AS history_years,
            cs.cancellation_count,
            cs.replacement_count,
            cs.rejection_count
            {label_sql}
        FROM latest_carriers cc
        JOIN company_stats cs USING (usdot)
    )
    SELECT *
    FROM joined
    {sample_sql}
    """

max_usable_snapshot_date = (
    pd.Timestamp(max_event_date) - pd.Timedelta(days=365)
).date()

# Limit training to recent years with a complete future 12-month window.
first_year = max(2021, max_usable_snapshot_date.year - 4)

snapshot_years = [
    year
    for year in range(first_year, max_usable_snapshot_date.year + 1)
    if date(year, 12, 31) <= max_usable_snapshot_date
]

print("\nHistorical snapshot years:", snapshot_years)

training_frames = []

for year in snapshot_years:
    cutoff = date(year, 12, 31)

    print(f"Building training snapshot for {cutoff}...")

    frame = con.execute(
        snapshot_query(cutoff, include_label=True, sample=True)
    ).df()

    print(
        f"  Rows: {len(frame):,}; "
        f"switch-label rate: "
        f"{frame['future_switch'].mean():.3f}"
        if len(frame) else "  No rows returned."
    )

    if len(frame):
        training_frames.append(frame)

if len(training_frames) < 2:
    raise ValueError(
        "Not enough historical snapshots with a complete 12-month "
        "outcome window. Check the event-date range and status filters."
    )

training = pd.concat(training_frames, ignore_index=True)

MODEL_FEATURES = [
    "event_count",
    "distinct_insurers",
    "days_since_last_placement",
    "history_years",
    "cancellation_count",
    "replacement_count",
    "rejection_count",
]

X_all = training[MODEL_FEATURES].replace([np.inf, -np.inf], np.nan)
y_all = training["future_switch"].astype(int)

if y_all.nunique() < 2:
    raise ValueError(
        "The historical snapshots contain only one target class. "
        "The change model cannot be trained until the event definition "
        "or snapshot range is revised."
    )

# Use the latest eligible snapshot year as a temporal holdout.
holdout_year = max(snapshot_years)
train_mask = training["snapshot_year"] < holdout_year
test_mask = training["snapshot_year"] == holdout_year

if train_mask.sum() == 0 or test_mask.sum() == 0:
    raise ValueError("Unable to construct a temporal validation split.")

model = HistGradientBoostingClassifier(
    max_iter=100,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    learning_rate=0.08,
    random_state=42,
)

model.fit(X_all.loc[train_mask], y_all.loc[train_mask])

test_probs = model.predict_proba(X_all.loc[test_mask])[:, 1]

if y_all.loc[test_mask].nunique() == 2:
    auc = roc_auc_score(y_all.loc[test_mask], test_probs)
    print(f"\nMODEL 2 TEMPORAL HOLDOUT ROC-AUC: {auc:.4f}")
else:
    print(
        "\nMODEL 2: Holdout has only one target class; "
        "ROC-AUC is unavailable."
    )

# Refit using all historical snapshots before scoring current companies.
model.fit(X_all, y_all)

print("\nMODEL 2 TRAINING LABEL RATE:", round(y_all.mean(), 4))

# ============================================================
# 6. SCORE COMPANIES AS OF THE LATEST OBSERVED EVENT DATE
# ============================================================

print("\nScoring companies for the next 12 months...")

current_snapshot = con.execute(
    snapshot_query(max_event_date, include_label=False, sample=False)
).df()

print("Companies with usable histories:", f"{len(current_snapshot):,}")

X_current = current_snapshot[MODEL_FEATURES].replace(
    [np.inf, -np.inf], np.nan
)

current_snapshot["change_probability"] = model.predict_proba(
    X_current
)[:, 1]

# ============================================================
# 7. COMBINE THE TWO SCORES
# ============================================================

fit_scores = con.execute("""
SELECT
    usdot,
    insurer AS observed_carrier,
    placement_date,
    carrier_company_count,
    fit_anomaly_score,
    fit_anomaly_percentile
FROM carrier_fit_scores
""").df()

results = current_snapshot.merge(
    fit_scores,
    on="usdot",
    how="inner",
)

# Exclude companies whose carrier profiles are too sparse to score.
results = results.dropna(subset=["fit_anomaly_percentile"]).copy()

# Initial hypothesis only: fit anomaly receives 60%, change
# probability receives 40%. Both are on a 0-1 scale.
results["combined_score"] = (
    0.60 * results["fit_anomaly_percentile"]
    + 0.40 * results["change_probability"]
)

results["priority_reason"] = np.select(
    [
        (results["fit_anomaly_percentile"] >= 0.8)
        & (results["change_probability"] >= 0.6),

        results["fit_anomaly_percentile"] >= 0.8,

        results["change_probability"] >= 0.6,
    ],
    [
        "High carrier anomaly and high change probability",
        "High carrier anomaly",
        "High insurance-change probability",
    ],
    default="Lower combined signal",
)

results = results.sort_values(
    "combined_score",
    ascending=False,
).reset_index(drop=True)

# Add company name and state where those columns exist.
label_columns = [
    c for c in ["company_name", "dba_name", "state"]
    if c in available_columns
]

if label_columns:
    label_selects = [
        f"ANY_VALUE({qi(c)}) AS {qi(c)}"
        for c in label_columns
    ]

    con.execute("DROP TABLE IF EXISTS company_labels")
    con.execute(f"""
        CREATE TABLE company_labels AS
        SELECT
            TRIM(CAST(usdot AS VARCHAR)) AS usdot,
            {", ".join(label_selects)}
        FROM raw
        WHERE usdot IS NOT NULL
        GROUP BY TRIM(CAST(usdot AS VARCHAR))
    """)

    labels = con.execute(
        "SELECT * FROM company_labels"
    ).df()

    results = results.merge(labels, on="usdot", how="left")

# Put the most useful columns first.
preferred_columns = [
    "usdot",
    "company_name",
    "dba_name",
    "state",
    "observed_carrier",
    "placement_date",
    "combined_score",
    "fit_anomaly_percentile",
    "fit_anomaly_score",
    "change_probability",
    "priority_reason",
    "carrier_company_count",
    *MODEL_FEATURES,
]

output_columns = [
    c for c in preferred_columns if c in results.columns
]

top_results = results[output_columns].head(TOP_N)

top_results.to_csv(OUTPUT_FILE, index=False)

print("\n================ FINAL RESULTS ================")
print(f"Companies ranked: {len(results):,}")
print(f"Saved top {len(top_results):,} to: {OUTPUT_FILE}")

print("\nTOP 25 COMPANIES")
print(top_results.head(25).to_string(index=False))

print("\nPRIORITY REASON COUNTS")
print(results["priority_reason"].value_counts().to_string())

print("\nSCORE SUMMARY")
print(
    results[
        [
            "fit_anomaly_percentile",
            "change_probability",
            "combined_score",
        ]
    ].describe().round(3).to_string()
)
